Export die to GDS (fractured to <=4000 vertices>)

In [1]:
import re
import gdsfactory as gf
import klayout.db as kdb
from amf.chp import PDK

gf.clear_cache()   # start from a clean cell registry (kills accumulated $N duplicates)
PDK.activate()

In [2]:
from myamf.Main_v2 import Main_v2

c = Main_v2()
c.plot()   # optional preview

/workspace/.venv/lib/python3.12/site-packages/gdsfactory/components/bends/bend_euler.py:112: UserWarning: {'width': 1.0} ignored for cross_section 'xs_aef91a05'
  x = gf.get_cross_section(cross_section, width=width or x.width)
/workspace/.venv/lib/python3.12/site-packages/gdsfactory/components/bends/bend_euler.py:112: UserWarning: {'width': 10.0} ignored for cross_section 'metal_routing'
  x = gf.get_cross_section(cross_section, width=width or x.width)
/workspace/.venv/lib/python3.12/site-packages/gdsfactory/components/bends/bend_euler.py:112: UserWarning: {'width': 0.5} ignored for cross_section 'strip'
  x = gf.get_cross_section(cross_section, width=width or x.width)
/workspace/.venv/lib/python3.12/site-packages/gdsfactory/components/bends/bend_euler.py:112: UserWarning: {'width': 1.0} ignored for cross_section 'nitride'
  x = gf.get_cross_section(cross_section, width=width or x.width)
/workspace/.venv/lib/python3.12/site-packages/gdsfactory/components/bends/bend_euler.py:112: UserWa

KeyError: "key='o1' is not a valid port name or index. Make sure the instance is an array when giving it a tuple. Available ports: []"

In [ ]:
# ---- Cell-name cleanup pass (operates on a KLayout layout) ----
ILLEGAL = re.compile(r"[^A-Za-z0-9_$?.]")     # GDS-safe character set
DUP     = re.compile(r"^(?P<base>.+)\$\d+$")  # KLayout conflict suffix "$N"

def clean_cell_names(layout: kdb.Layout) -> None:
    # 1) Merge "<name>$N" duplicates into canonical "<name>" (or restore the exact
    #    bb name when only the suffixed copy exists). Transforms/arrays preserved.
    changed = True
    while changed:
        changed = False
        for cell in list(layout.each_cell()):
            m = DUP.match(cell.name)
            if not m:
                continue
            base = m.group("base")
            if layout.has_cell(base) and layout.cell_by_name(base) != cell.cell_index():
                canon = layout.cell(layout.cell_by_name(base))
                for caller_idx in list(cell.caller_cells()):
                    caller = layout.cell(caller_idx)
                    for inst in list(caller.each_inst()):
                        if inst.cell_index == cell.cell_index():
                            ci = inst.cell_inst          # full CellInstArray
                            ci.cell_index = canon.cell_index()
                            caller.insert(ci)
                            caller.erase(inst)
                layout.delete_cell(cell.cell_index())
                changed = True
                break
            elif not layout.has_cell(base):
                cell.name = base                          # strip "$N" -> exact bb name
                changed = True
                break

    # 2) Strip illegal characters (e.g. "<defunct>$2" -> "defunct$2")
    seen = set(cl.name for cl in layout.each_cell())
    for cell in layout.each_cell():
        if ILLEGAL.search(cell.name):
            new = ILLEGAL.sub("", cell.name) or "cell"
            while new in seen and new != cell.name:
                new += "_x"
            seen.discard(cell.name); seen.add(new)
            cell.name = new

In [ ]:
# ---- Export pipeline ----
TMP   = "/workspace/myamf/gds/_tmp_main.gds"
FINAL = "/workspace/myamf/gds/Main_AMF_Die_clean.gds"

# Stage 1: write WITHOUT metadata -> removes gdsfactory port/pin layers from bb cells
c.write_gds(TMP, with_metadata=False)

# Stage 2: clean cell names, then write final with polygons fractured to <= 4000 verts
layout = kdb.Layout()
layout.read(TMP)
clean_cell_names(layout)

opts = kdb.SaveLayoutOptions()
opts.gds2_max_vertex_count = 4000
opts.gds2_multi_xy_records = False
layout.write(FINAL, opts)
print("Wrote:", FINAL)

In [ ]:
# ---- Verify the submission file ----
chk = kdb.Layout()
chk.read(FINAL)

bad_names = [c.name for c in chk.each_cell()
             if re.search(r"\$\d+$", c.name) or ILLEGAL.search(c.name)]
worst = 0
for cell in chk.each_cell():
    for li in chk.layer_indexes():
        for s in cell.shapes(li).each():
            p = s.polygon
            if p:
                worst = max(worst, p.num_points_hull())

print("cells with $N or illegal chars:", bad_names if bad_names else "none")
print("max polygon vertices:", worst)
assert not bad_names, "Still have misnamed/illegal cells!"
assert worst <= 4000, "Still have polygons over 4000 vertices!"
print("OK - names clean, all polygons <= 4000 vertices. Safe to submit.")